# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umaimakhalid17/ML/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

**Final score**, following the exact blend pattern the lane guide describes for FlyRank's own
final score — model probability leads, the transparent baseline anchors it:

```text
final_action_score = 100 * (0.70 * model_probability + 0.30 * normalized_baseline_score)
```

`model_probability` is `w06`'s honest logistic regression (client-grouped AUC 0.604), trained
here on the full starter slice for production-style ranking. `normalized_baseline_score` is
`w04`'s rule, scaled 0–1. Reason codes (priority order, one per row — the highest-priority
condition that fires wins):

| `reason_code` | condition | `action_label` |
|---|---|---|
| `model_decline_risk` | model probability ≥ 0.65 | `priority_refresh` |
| `ctr_review_candidate` | visible (≥500 impr, position 1–20) and `ctr < 0.5` | `review_ctr_and_meta` |
| `engagement_review_candidate` | `sessions_90d ≥ 30` and engagement/scroll rate < 30 | `review_content_depth` |
| `visible_model_opportunity` | model probability ≥ 0.50 and `impressions_90d ≥ 500` | `light_touch_review` |
| *(none of the above)* | — | `monitor` |

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import os

np.random.seed(42)
pd.set_option("display.width", 140)

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

eligible = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)
ctr_gap = eligible & (df["ctr"] < 0.5)
severity = (0.5 - df["ctr"]).clip(lower=0)
df["baseline_action_score"] = np.where(ctr_gap, np.log1p(df["impressions_90d"]) * severity, 0.0)
bmax = df["baseline_action_score"].max()
df["baseline_norm"] = df["baseline_action_score"] / bmax if bmax > 0 else 0.0

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
categorical_features = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]
missingness_flag_cols = ["search_volume", "competition", "cpc", "word_count", "char_count"]

X = df[numeric_features + categorical_features].copy()
for c in missingness_flag_cols:
    X[f"has_{c}"] = df[c].notna().astype(int)
X[numeric_features] = X[numeric_features].fillna(0)
X[categorical_features] = X[categorical_features].fillna("unknown")
num_cols_final = numeric_features + [f"has_{c}" for c in missingness_flag_cols]

y = df["is_declining_label"].values
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols_final),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

# Production-style fit on the full slice for ranking. Every metric this playbook and the paper
# QUOTE comes from w05/w06's held-out, client-grouped evaluation -- never from this fit.
model = Pipeline([("pre", preprocess), ("clf", LogisticRegression(max_iter=3000, random_state=42))])
model.fit(X, y)
df["model_proba"] = model.predict_proba(X)[:, 1]

df["final_action_score"] = 100 * (0.70 * df["model_proba"] + 0.30 * df["baseline_norm"])

model_decline_risk = df["model_proba"] >= 0.65
engagement_review = (df["sessions_90d"] >= 30) & (
    ((df["engagement_rate"] > 0) & (df["engagement_rate"] < 30)) |
    ((df["scroll_rate"] > 0) & (df["scroll_rate"] < 30))
)
visible_model_opp = (df["model_proba"] >= 0.50) & (df["impressions_90d"] >= 500)

df["reason_code"] = np.select(
    [model_decline_risk, ctr_gap, engagement_review, visible_model_opp],
    ["model_decline_risk", "ctr_review_candidate", "engagement_review_candidate", "visible_model_opportunity"],
    default="monitor",
)
action_map = {
    "model_decline_risk": "priority_refresh",
    "ctr_review_candidate": "review_ctr_and_meta",
    "engagement_review_candidate": "review_content_depth",
    "visible_model_opportunity": "light_touch_review",
    "monitor": "monitor",
}
df["action_label"] = df["reason_code"].map(action_map)

queue = df.sort_values("final_action_score", ascending=False).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)

print("Reason code counts:")
print(queue["reason_code"].value_counts())
print()
print("Decline rate by reason_code (lift check vs. the 54.2% base rate):")
print(queue.groupby("reason_code")["is_declining_label"].mean().round(3))
print()
cols = ["rank", "content_id", "client_id", "final_action_score", "model_proba", "reason_code",
        "action_label", "impressions_90d", "avg_position", "ctr", "is_declining_label"]
print("Top 5 of the ranked queue:")
print(queue[cols].head(5).to_string(index=False))


Reason code counts:
reason_code
monitor                        11472
model_decline_risk              9643
ctr_review_candidate            5343
engagement_review_candidate     2458
visible_model_opportunity       1084
Name: count, dtype: int64

Decline rate by reason_code (lift check vs. the 54.2% base rate):
reason_code
ctr_review_candidate           0.534
engagement_review_candidate    0.470
model_decline_risk             0.737
monitor                        0.399
visible_model_opportunity      0.524
Name: is_declining_label, dtype: float64

Top 5 of the ranked queue:
 rank           content_id         client_id  final_action_score  model_proba        reason_code     action_label  impressions_90d  avg_position  ctr  is_declining_label
    1 content_453722754fea client_f369cb89fc           88.155294     0.853029 model_decline_risk priority_refresh           140079           7.6 0.01                   1
    2 content_f986bd514b6e client_7f2253d7e2           86.443971     0.884341 model_

**Reading the queue as a human would trust it:** `model_decline_risk` rows show a 73.7% observed
decline rate against a 54.2% base rate — the strongest reason code by lift, and it fires on 9,643
of 30,000 pages (too many for one editor's week, but this is a ranked queue, not a to-do list —
the score decides the order within that group). `ctr_review_candidate` and
`visible_model_opportunity` sit closer to the base rate, which is honest: they flag genuine
opportunity, not certain decline.

## 2. Intended use and limits

**Who uses this:** a content editor or SEO lead triaging a weekly refresh queue for one client's
site — the same persona named back in `w01`.

**What it's for:** deciding which handful of pages to open first out of hundreds, with a reason
attached, so the editor can sanity-check the call before spending time on it.

**Where it stops being valid:**
- **A new client with no history in this slice.** The model was trained on this portfolio; `w06`
  showed a real 0.09 AUC gap between random and client-grouped evaluation, meaning performance on
  a genuinely new client will likely sit closer to the honest 0.604 AUC than any better-looking
  number.
- **Anything framed as "this refresh will cause growth."** Every score here is a **directional,
  decision-support** ranking over an observational proxy label (`trend_direction`), never a
  causal estimate — `w06` Section 1 shows why the FlyRank paper's own strongest refresh claim
  needs the same caution.
- **Populations this dataset excludes.** Every row here already has `impressions_90d > 0` and
  `content_age_days >= 90` (the starter CSV's own construction rule) — brand-new pages and pages
  with zero search visibility are outside this playbook's scope entirely, not scored as "safe."
- **Beyond the 30,000-row starter slice.** This is a teaching-scale sample of a ~79M-row
  warehouse; treat every number here as "observed in this slice," not "true of FlyRank clients
  in general."

In [2]:
print("Rows this playbook can score (impressions_90d > 0, content_age_days >= 90):", len(df))
print("Reminder: starter slice size 30,000 rows vs. ~79M-row full warehouse -- scope stated,")
print("not just implied.")


Rows this playbook can score (impressions_90d > 0, content_age_days >= 90): 30000
Reminder: starter slice size 30,000 rows vs. ~79M-row full warehouse -- scope stated,
not just implied.


## 3. Human review + the no-go list

**What a person must check before acting on any row:**
- Confirm the page's current CTR/position isn't a **tracking or tagging artifact** —
  `w05` Section 4 found exactly this pattern in two of its wrong cases (near-zero CTR on
  six-figure impression counts is at least as consistent with a measurement bug as a real
  content problem).
- Check for **client-level concentration** in the top of the queue — `w04`'s top-10 review found
  3–4 rows from the same `client_id`, which can mean "this client's tracking setup" rather than
  ten independent content problems.
- Read the actual page before editing anything — `main_intent` (e.g. navigational/branded
  queries) can make "low CTR" mean "users already know the URL," not "the snippet is bad."

**The no-go list — never automate these:**
- **No automatic content edits, publishing, or deletion.** Every row is a suggestion an editor
  opens and judges, exactly as `w01`'s decision framing set out — never a script that rewrites
  or removes a page on its own.
- **No claim of causing recovery.** Nothing here is licensed to say "refreshing this page will
  increase traffic by X%" — see the Finding #4 critique in `w06`.
- **No cross-client scoring without re-validation.** Don't apply this fitted model to a brand-new
  client's pages and trust the score at face value; retrain or re-validate first (Section 4).

## 4. Monitoring / retrain triggers

Signals that would tell me this queue has gone stale, and what I'd do about each:

| Trigger | What it would look like | Response |
|---|---|---|
| **Precision drift** | Precision@50 on a fresh month's data drops well below the 0.604 AUC / 0.76 precision@50 benchmark from `w06` | Re-run `w06`'s comparison table on new data before trusting the queue again |
| **Reason-code mix shifts sharply** | `model_decline_risk` share jumps far past its current ~32% of rows, or collapses toward 0 | Investigate a possible upstream data or tracking change before assuming the portfolio itself changed that fast |
| **New client onboarded** | A `client_id` with zero rows in this training slice appears in the queue | Treat its scores as unvalidated (Section 2) until at least one grouped-holdout check is run on it specifically |
| **Base rate moves** | The 54.2% overall decline rate shifts materially quarter over quarter | Recompute the base rate before comparing any precision number back to this notebook's table — a stale base rate makes lift look fake either direction |
| **Time passes without retraining** | No retrain in 2+ reporting cycles | Re-run this notebook end-to-end on the latest slice as a fixed cadence, not only when something looks wrong |

In [3]:
# No new computation -- this section is the monitoring plan itself. One quick number worth
# keeping on record as the "current" baseline to watch for drift against:
print("Current reference numbers to monitor against (from w06, client-grouped holdout):")
print("  AUC: 0.604")
print("  Precision@50: 0.76  |  Precision@100: 0.69  |  Precision@250: 0.712")
print("  Base rate (decline): 0.542 (full slice) / 0.559 (this holdout fold)")


Current reference numbers to monitor against (from w06, client-grouped holdout):
  AUC: 0.604
  Precision@50: 0.76  |  Precision@100: 0.69  |  Precision@250: 0.712
  Base rate (decline): 0.542 (full slice) / 0.559 (this holdout fold)


## 5. Exports for the paper

Writing the final ranked queue and a couple of reusable summary artifacts to `work/outputs/` —
the paper's Results and Ranked Recommendations sections build on these files.

In [4]:
os.makedirs("../outputs", exist_ok=True)

output_cols = ["rank", "content_id", "client_id", "final_action_score", "model_proba",
               "baseline_norm", "reason_code", "action_label", "impressions_90d", "avg_position",
               "ctr", "days_since_last_update", "word_count", "content_type", "is_declining_label"]
queue_path = "../outputs/final_action_queue.csv"
queue[output_cols].to_csv(queue_path, index=False)
print(f"Wrote final ranked queue: {queue_path} ({len(queue)} rows)")

import json as _json
playbook_metrics = {
    "rows": int(len(queue)),
    "reason_code_counts": queue["reason_code"].value_counts().to_dict(),
    "decline_rate_by_reason_code": {
        k: round(float(v), 4)
        for k, v in queue.groupby("reason_code")["is_declining_label"].mean().items()
    },
    "decline_rate_overall": round(float(df["is_declining_label"].mean()), 4),
    "score_formula": "100 * (0.70 * model_proba + 0.30 * baseline_norm)",
    "held_out_reference_metrics": {
        "grouped_split_auc": 0.604,
        "random_split_auc": 0.694,
        "precision_at_50": {"baseline": 0.62, "logistic_regression": 0.76, "random_forest": 0.54},
        "precision_at_100": {"baseline": 0.67, "logistic_regression": 0.69, "random_forest": 0.57},
        "precision_at_250": {"baseline": 0.676, "logistic_regression": 0.712, "random_forest": 0.628},
    },
}
with open("../outputs/playbook_metadata.json", "w") as f:
    _json.dump(playbook_metrics, f, indent=2)
print("Wrote playbook metrics: ../outputs/playbook_metadata.json")


Wrote final ranked queue: ../outputs/final_action_queue.csv (30000 rows)
Wrote playbook metrics: ../outputs/playbook_metadata.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.